In [1]:
#reading data
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
import numpy as np
raw = pd.read_csv("../data/campaigns_messy.csv")
brands=pd.read_csv("../data/brands.csv")

In [2]:
#problem 1: col names not clean
#cleaning column names

raw.columns = [col.strip().lower() for col in raw.columns]
raw= raw.rename(columns=
                {
                    'campaign id' : 'campaign_id',
                    'start date': 'start_date',
                    'budget (usd)': 'budget_usd'
                    
                })

#checking dtypes and formats
raw.shape
[raw[col].unique() for col in raw.columns]

(155, 12)

[<ArrowStringArray>
 ['CMP126', 'CMP098', 'CMP035', 'CMP026', 'CMP020', 'CMP070', 'CMP106',
  'CMP013', 'CMP046', 'CMP093',
  ...
  'CMP095', 'CMP110', 'CMP104', 'CMP059', 'CMP138', 'CMP051', 'CMP088',
  'CMP105', 'CMP130', 'CMP123']
 Length: 150, dtype: str,
 <ArrowStringArray>
 [    'SNACKBOX', 'Urban Thread',      'fitfuel',      ' GLOWLY',
      'homenest',      'FitFuel',    'HOMENEST ',       'Glowly',
      'snackbox',     'SnackBox',     'HomeNest',     'FITFUEL ',
   'urbanthread',  'UrbanThread',     'Fit Fuel',       'glowly',
       'Glowly ',    'Snack Box']
 Length: 18, dtype: str,
 <ArrowStringArray>
 ['instagram',        'FB',    'TikTok', 'Instagram',        'IG',     'Insta',
     'TIKTOK',  'Facebook',   'Tik Tok',    'tiktok',  'facebook']
 Length: 11, dtype: str,
 <ArrowStringArray>
 ['2026-03-23', '2026-02-26', '2026-05-23', '2026-02-13', '2026-06-08',
  '2026-01-12', '2026-05-20', '2026-04-07', '2026-01-16', '2026-02-19',
  ...
  '2026-02-05', '11/04/2026', '2026

In [3]:
#problem 2: duplicate rows
#columnwise cleaning
len(raw['campaign_id'])== len(raw['campaign_id'].unique())
raw_v1 = raw.drop_duplicates(subset=['campaign_id'], keep='first')
len(raw_v1['campaign_id'])== len(raw_v1['campaign_id'].unique())
raw_v1.info()

False

True

<class 'pandas.DataFrame'>
Index: 150 entries, 0 to 154
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   campaign_id  150 non-null    str    
 1   brand        150 non-null    str    
 2   platform     150 non-null    str    
 3   start_date   150 non-null    str    
 4   end_date     150 non-null    str    
 5   budget_usd   150 non-null    str    
 6   spend        149 non-null    float64
 7   impressions  149 non-null    str    
 8   clicks       150 non-null    int64  
 9   conversions  150 non-null    int64  
 10  status       146 non-null    str    
 11  manager      129 non-null    str    
dtypes: float64(1), int64(2), str(9)
memory usage: 24.6 KB


In [4]:
#problem 3: brands nott unified
raw_v1['brand']=raw_v1['brand'].str.strip().str.lower()
raw_v1['brand']= raw_v1['brand'].replace({'urban thread': 'UrbanThread', 
                                         'urbanthread': 'UrbanThread',
                                         'snackbox': 'SnackBox',
                                         'snack box': 'SnackBox',
                                         'fitfuel': 'FitFuel',
                                        'fit fuel': 'FitFuel',
                                         'homenest': 'HomeNest',
                                         'glowly': 'Glowly'})
raw_v1['brand'].unique()
raw_v2= raw_v1.copy()

<ArrowStringArray>
['SnackBox', 'UrbanThread', 'FitFuel', 'Glowly', 'HomeNest']
Length: 5, dtype: str

In [5]:
#problem 4: platforms named differently
raw_v2['platform']= raw_v2['platform'].str.strip().str.lower()
raw_v2['platform']= raw_v2['platform'].replace({'instagram': 'Instagram', 
                                         'ig': 'Instagram',
                                         'insta': 'Instagram',       
                                         'fb': 'Facebook',
                                         'facebook': 'Facebook',
                                         'tiktok': 'TikTok',
                                        'tik tok': 'TikTok'})
raw_v2['platform'].unique()
raw_v3= raw_v2.copy()

<ArrowStringArray>
['Instagram', 'Facebook', 'TikTok']
Length: 3, dtype: str

In [6]:
#problem 5: dates have / in them which is causing conversion error
# Force pandas to prioritize Month-Day-Year for ambiguous dates
for col in ['start_date', 'end_date']:
    iso_parsed = pd.to_datetime(raw_v3[col], format='%Y-%m-%d', errors='coerce')
    mdy_parsed = pd.to_datetime(raw_v3[col], format='mixed', dayfirst=False, errors='coerce')
    raw_v3[col] = iso_parsed.fillna(mdy_parsed).dt.strftime('%Y-%m-%d')
raw_v4 = raw_v3.copy()

In [7]:
#problem 6: currencies entered in diff formats
raw_v4['budget_usd'] = raw_v4['budget_usd'].replace(r'[\$,]', '', regex=True)
raw_v4['budget_usd'] = raw_v4['budget_usd'].str.replace('k', '000')
raw_v4['budget_usd'].unique()
raw_v5= raw_v4.copy()

<ArrowStringArray>
['8000', '5000', '15000', '25000', '3000', '20000', '2000', '12000', '10000']
Length: 9, dtype: str

In [8]:
#spend has negatives
raw_v5['spend'].describe()
raw_v6= raw_v5.copy()
# raw_v5.loc[raw_v5['spend'] < 0, 'spend'] = raw_v5.loc[df['spend'] < 0, 'spend'].abs()

count      149.000000
mean      9203.030201
std       6873.193479
min     -17408.480000
25%       4053.560000
50%       8548.390000
75%      13597.930000
max      28050.890000
Name: spend, dtype: float64

In [9]:
#problem 7: impressions have '-' characters
raw_v6['impressions']=raw_v6['impressions'].replace('-', np.nan)
raw_v6['impressions'].unique()
raw_v7= raw_v6.copy()

<ArrowStringArray>
[ '544735',  '657621', '4184784', '1213095',  '792390',  '628003', '1085683',
       nan,  '702981', '4327767',
 ...
  '735220',  '486485',  '251062', '1623931',  '304347', '2444841', '1726409',
 '1923407', '2780890',  '523824']
Length: 148, dtype: str

In [10]:
#problem 8: status needs cleaning
raw_v7['status']=raw_v7['status'].str.strip().str.lower().str.replace('completed', 'complete')
raw_v7['status'].unique()
raw_v8= raw_v7.copy()

<ArrowStringArray>
['complete', 'active', 'paused', nan]
Length: 4, dtype: str

In [11]:
# problem 9: converting dtypes
raw_v8 = raw_v8.astype({'campaign_id': str, 'brand': str, 'platform': str, 'start_date': 'datetime64[ns]', 'end_date': 'datetime64[ns]', 'budget_usd': float, 'spend': float, 'impressions': 'Int64', 'clicks': 'int64', 'conversions': 'int64', 'status': 'category', 'manager': str})
raw_v9=raw_v8.copy()

In [12]:
#problem 10: missing values

raw_v9.isna().sum()

# raw_v8[raw_v8['spend'].isna()] 
raw_v9['spend'] = raw_v9['spend'].fillna(
    raw_v9.groupby(['brand', 'platform'])['spend'].transform('median')
)

# raw_v8[raw_v8['impressions'].isna()] 
raw_v9['impressions'] = raw_v9['impressions'].fillna(
    raw_v9.groupby(['brand', 'platform'])['impressions'].transform('median')
)

#no need to fill status as it's only 4 unknowns and we don't need it for analysis

raw_v9['manager']= raw_v9['manager'].fillna('Unknown')

raw_v10=raw_v9.copy()

campaign_id     0
brand           0
platform        0
start_date      0
end_date        0
budget_usd      0
spend           1
impressions     3
clicks          0
conversions     0
status          4
manager        21
dtype: int64

In [13]:
#proving data is clean
raw_v10.shape #5 dupes dropped
raw_v10.info() #dtypes fixed
raw_v10.isna().sum() #missing values dealth with

(150, 12)

<class 'pandas.DataFrame'>
Index: 150 entries, 0 to 154
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   campaign_id  150 non-null    str           
 1   brand        150 non-null    str           
 2   platform     150 non-null    str           
 3   start_date   150 non-null    datetime64[ns]
 4   end_date     150 non-null    datetime64[ns]
 5   budget_usd   150 non-null    float64       
 6   spend        150 non-null    float64       
 7   impressions  150 non-null    Int64         
 8   clicks       150 non-null    int64         
 9   conversions  150 non-null    int64         
 10  status       146 non-null    category      
 11  manager      150 non-null    str           
dtypes: Int64(1), category(1), datetime64[ns](2), float64(2), int64(2), str(4)
memory usage: 18.3 KB


campaign_id    0
brand          0
platform       0
start_date     0
end_date       0
budget_usd     0
spend          0
impressions    0
clicks         0
conversions    0
status         4
manager        0
dtype: int64

In [14]:
import pandas as pd
import numpy as np

def clean_campaigns(path):
    # 1. Load data
    raw = pd.read_csv(path)
    
    # 2. Chain column cleaning, renaming, and deduplication
    raw.columns = [col.strip().lower() for col in raw.columns]
    raw = (raw.rename(columns={
                  'campaign id': 'campaign_id', 
                  'start date': 'start_date', 
                  'budget (usd)': 'budget_usd'
              })
              .drop_duplicates(subset=['campaign_id'], keep='first'))
    
    # 3. Chain brand name normalization
    raw['brand'] = raw['brand'].str.strip().str.lower().replace({
        'urban thread': 'UrbanThread', 'urbanthread': 'UrbanThread',
        'snackbox': 'SnackBox', 'snack box': 'SnackBox',
        'fitfuel': 'FitFuel', 'fit fuel': 'FitFuel',
        'homenest': 'HomeNest', 'glowly': 'Glowly'
    })
    
    # 4. Chain platform name normalization
    raw['platform'] = raw['platform'].str.strip().str.lower().replace({
        'instagram': 'Instagram', 'ig': 'Instagram', 'insta': 'Instagram',       
        'fb': 'Facebook', 'facebook': 'Facebook',
        'tiktok': 'TikTok', 'tik tok': 'TikTok'
    })
    
    # 5. Dates: ISO first, then UK day/month/year (the data has '24/01/2026', so day comes first)
    for col in ['start_date', 'end_date']:
        iso = pd.to_datetime(raw[col], format='%Y-%m-%d', errors='coerce')
        uk = pd.to_datetime(raw[col], format='%d/%m/%Y', errors='coerce')
        raw[col] = iso.fillna(uk)

    # 6. Clean currency and flag questionable metrics

    raw['budget_usd'] = (raw['budget_usd']
                         .str.replace(r'[\$,]', '', regex=True)
                         .str.replace('k', '000'))
    raw['spend_was_missing'] = raw['spend'].isna()
    raw['spend_was_negative'] = raw['spend'] < 0
    raw.loc[raw['spend_was_negative'], 'spend'] = (
        raw.loc[raw['spend_was_negative'], 'spend'].abs()
    )
    raw['impressions_was_missing'] = (
        raw['impressions'].isna() | (raw['impressions'] == '-')
    )
    raw['impressions'] = raw['impressions'].replace('-', np.nan)
    
    raw['status'] = raw['status'].str.strip().str.lower().replace(
        'complete', 'completed'
    )
    
    # 7. Convert types across the entire dataframe
    raw = raw.astype({
    'campaign_id': str,
    'brand': str,
    'platform': str,
    'budget_usd': float,
    'spend': float,
    'impressions': 'Int64',
    'clicks': 'int64',
    'conversions': 'int64',
    'status': 'category',
    'manager': str})
    
    # 8. Impute missing values using group medians
    raw['manager'] = raw['manager'].fillna('Unknown')

    # 9. Impossible rows: flag, then drop (can't be trusted for any metric)
    raw['impossible_clicks'] = (raw['clicks'] > raw['impressions']).fillna(False)
    raw['impossible_dates'] = raw['end_date'] < raw['start_date']
    raw = raw[~(raw['impossible_clicks'] | raw['impossible_dates'])]
    
    return raw


In [15]:
#Checking if funtion is working
clean_campaigns("../data/campaigns_messy.csv").equals(clean_campaigns("../data/campaigns_messy.csv"))

True

In [16]:
#Loading it as a proper dataset for analysis
df= clean_campaigns("../data/campaigns_messy.csv")
df.shape, raw.shape #5 duplicate rows removed
df['brand'].unique()
df['platform'].unique()
df['status'].unique()
df.dtypes
df.isna().sum() #Nans filled for numeric rows except status, because it is not required for analysis
df['budget_usd'].describe() 
df['spend'].describe()
df['impressions'].describe()
df['clicks'].describe()
df['conversions'].describe()
df['clicks_over_impressions']= df['clicks']>df['impressions']
df['date_check']= df['end_date'] < df['start_date'] 
df[df['clicks_over_impressions']]
#all make sense now

df['spend'].isna().sum()
df['spend_was_negative'].sum()
df['impressions_was_missing'].sum()
df['start_date'].isna().sum()
df['end_date'].isna().sum()
df.loc[df['date_check']==True] #included in analysis-- this could be entry errors or maybe something i did in cleaning
#issues too little to affect analysis

((147, 17), (155, 12))

<ArrowStringArray>
['SnackBox', 'UrbanThread', 'FitFuel', 'Glowly', 'HomeNest']
Length: 5, dtype: str

<ArrowStringArray>
['Instagram', 'Facebook', 'TikTok']
Length: 3, dtype: str

['completed', 'active', 'paused', NaN]
Categories (3, str): ['active', 'completed', 'paused']

campaign_id                           str
brand                                 str
platform                              str
start_date                 datetime64[us]
end_date                   datetime64[us]
budget_usd                        float64
spend                             float64
impressions                         Int64
clicks                              int64
conversions                         int64
status                           category
manager                               str
spend_was_missing                    bool
spend_was_negative                   bool
impressions_was_missing              bool
impossible_clicks                 boolean
impossible_dates                     bool
dtype: object

campaign_id                0
brand                      0
platform                   0
start_date                 0
end_date                   0
budget_usd                 0
spend                      1
impressions                3
clicks                     0
conversions                0
status                     4
manager                    0
spend_was_missing          0
spend_was_negative         0
impressions_was_missing    0
impossible_clicks          0
impossible_dates           0
dtype: int64

... (8 outputs left out)

np.int64(3)

np.int64(0)

np.int64(0)

,campaign_id,brand,platform,start_date,end_date,budget_usd,spend,impressions,clicks,conversions,status,manager,spend_was_missing,spend_was_negative,impressions_was_missing,impossible_clicks,impossible_dates,clicks_over_impressions,date_check


In [24]:
# D1. Budget use by brand

# (a) Total budget, total spend and % of budget spent, per brand.
budget_by_brand= df.groupby('brand').agg(
    total_budget= ('budget_usd', 'sum'),
    total_spend= ('spend', 'sum')
)
budget_by_brand['%_spent']= (budget_by_brand['total_spend']/budget_by_brand['total_budget'])*100
# (b) Round the % to 1.
budget_by_brand.round(1).sort_values('%_spent', ascending=False)
# (c) Sort highest % first.

,total_budget,total_spend,%_spent
brand,,,
Glowly,368000.0,315868.6,85.8
UrbanThread,296000.0,236580.2,79.9
SnackBox,350000.0,278384.4,79.5
HomeNest,387000.0,305588.6,79.0
FitFuel,339000.0,263813.9,77.8


In [18]:
# D2. Click-through rate (CTR) by platform

# (a) CTR = clicks / impressions × 100, per platform.

df['ctr_individual']= (df['clicks']/df['impressions'])
method_1= (df.groupby('platform')['ctr_individual'].mean()*100).round(3)
method_1.reset_index()

method_2 = (
    df.groupby('platform')
    .agg(
        total_clicks=('clicks', 'sum'),
        total_impressions=('impressions', 'sum')
    )
    .reset_index()
)

method_2['true_ctr'] = ((method_2['total_clicks'] / method_2['total_impressions'])*100).round(3)
method_2

# (b) Round to 3.
# (c) Comment: there are two ways to compute a platform’s CTR: the average of each campaign’s CTR, or total clicks / total impressions. Which did you use, and why?
# Total clicks ÷ total impressions weights big campaigns more, because every impression counts once. The average of ratios gives a tiny campaign the same vote as a huge one. That’s why the pooled version is the true platform CTR.

,platform,ctr_individual
0,Facebook,0.851
1,Instagram,1.227
2,TikTok,2.011


,platform,total_clicks,total_impressions,true_ctr
0,Facebook,461510,54457521,0.847
1,Instagram,1234989,93844904,1.316
2,TikTok,1520201,79512892,1.912


In [19]:
# D3. Most cost-efficient campaigns

# (a) Cost per conversion = spend / conversions.
df['cost_per_conversion']= df['spend']/df['conversions']
# (b) Comment: what problem appears for some campaigns, and how did you handle it?
df['conversions_not_defined']= df['conversions']==0
# 3 campaigns
#infinity cost, because there were no conversions at all as per the data
#might have raised some issues for the ones which were negative spend, but i converted them to positive earlier so i do not see any issue there.
# (c) Show the top 5 (lowest cost per conversion): campaign_id, brand, platform and the cost.
df.sort_values('cost_per_conversion')[['campaign_id', 'brand', 'platform', 'cost_per_conversion']].head(5)

,campaign_id,brand,platform,cost_per_conversion
126,CMP147,SnackBox,TikTok,3.453584
147,CMP104,Glowly,TikTok,3.547464
68,CMP040,Glowly,TikTok,3.741111
125,CMP029,FitFuel,TikTok,4.173690
37,CMP101,FitFuel,TikTok,4.265958


In [20]:
# D4. Spend over time

# (a) Total spend per month (by start date).
# Resample by Month Start ('MS') and get the total spend
monthly_table = df.resample('MS', on='start_date').agg(total_spend=('spend', 'sum'))
monthly_table
# (b) Which month had the most spend?
monthly_table.loc[monthly_table.idxmax()] #april

,total_spend
start_date,
2026-01-01,277041.70
2026-02-01,182283.37
2026-03-01,190199.32
2026-04-01,306337.19
2026-05-01,205156.89
2026-06-01,239217.11


,total_spend
start_date,
2026-04-01,306337.19


In [21]:
# D5. Over budget

# (a) How many campaigns spent more than their budget?
df['over_budget']= df['spend']>df['budget_usd']
df['over_budget'].sum()

# (b) Which brand has the most over-budget campaigns?
df[df['over_budget'] == True].groupby('brand').size().sort_values(ascending=False)
#UrbanThread

np.int64(31)

brand
UrbanThread    10
Glowly          7
HomeNest        5
SnackBox        5
FitFuel         4
dtype: int64

In [23]:
# D6. Category view

# (a) Merge with brands.csv to add category.
merged=df.merge(brands, on='brand', how='left')
# (b) Check the row counts before and after.
df.shape, merged.shape
# (c) Total spend per category.
merged['category']= merged['category'].fillna('Unknown')
merged.groupby('category')['spend'].sum().reset_index()
# (d) Comment: which brand has no category, and how did you handle it?
#HomeNest has no category, so it was defined as 'Unknown' to esnure analysis remains correct.

((147, 23), (147, 27))

,category,spend
0,Unknown,305588.58
1,beauty,315868.56
2,fashion,236580.19
3,fitness,263813.88
4,food,278384.37


| # | Problem | What I did | Why | Rows affected |
|---|---|---|---|---|
| 1 | Messy column names (spaces, caps, "Budget (USD)") | Stripped, lowercased, renamed to snake_case | Consistent, typeable names | All columns |
| 2 | Duplicate campaigns | Dropped repeats of campaign_id | Would double-count spend and budget | 5 rows (155 → 150) |
| 3 | Same brand / platform / status written many ways ("Snack Box", "IG", "COMPLETED") | strip + lower, then map to official names | Otherwise one brand splits into several groups | brand 18 → 5, platform 11 → 3, status 9 → 3 |
| 4 | Dates in two formats (ISO and dd/mm/yyyy) | Parsed each format separately, combined | Mixed parsing swapped day/month and broke the timeline | 63 rows had at least one dd/mm/yyyy date |
| 5 | Budget as text ("$8,000", "8k", "5000") | Removed $ and commas, k → 000, converted to number | Can't sum or compare text | 150 rows (27 used "k") |
| 6 | Hidden missing impressions ("-") | Converted to NaN, flagged, left unfilled | Filling would invent impressions and distort CTR | 3 rows |
| 7 | Missing spend | Flagged, left as NaN | Filling would invent money; sums skip NaN | 1 row |
| 8 | Negative spend | Flagged, converted to positive | Most likely a sign-entry error; flag keeps it traceable | 2 rows |
| 9 | Clicks > impressions | Dropped | Physically impossible; distorted CTR (TikTok 3.56% → 1.91%) | 2 rows |
| 10 | End date before start date | Dropped | Impossible campaign timeline | 1 row (150 → 147 total after 9 and 10) |
| 11 | Missing status | Left as NaN | No reliable rule to infer it, and no analysis needs it | 4 rows |
| 12 | Missing manager | Filled "Unknown" | Honest label, not used in analysis | 21 rows |
| 13 | Zero conversions | Excluded from cost per conversion | spend ÷ 0 = infinity; cost isn't defined | 3 rows |
| 14 | HomeNest not in brands.csv | Category set to "Unknown" | We don't know its category, so we don't guess | 34 rows in D6 |

The overall budget spent was $1.40M out of $1.74M budget (about 80%), with Glowly utilizing the maximum budget (85.8%) followed by UrbanThread 79.9%, SnackBox 79.5%, HomeNest 79.0%. FitFuel 77.8% (lowest) used the lowert of its budget.
TikTok succeeds in CTR, having it at 1.91%, then came Instagram at 1.32% and then Facebook at 0.85%.
All cost-efficient campaigs were run on TikTok: Best is CMP147 (SnackBox) at $3.45 per conversion. However, 3 campaigns had no conversions at all.
The top month for spending was April ($306K) by the month campaigns started.
31 of 147 campaigns went over budget. UrbanThread had the most (10).
Beauty had the highest spend ($316K). HomeNest’s $306K has no category in our brand list though, and being 2nd in the list, this needs to be defined. 
TikTok is our best-performing platform on both clicks and cost per conversion, so it’s worth shifting more budget there”, or “UrbanThread’s 10 overspends suggest its budgets need tighter monitoring.